In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import pandas as pd


DEFAULT_DATASET_ROOT = Path(
    "/kaggle/input/datasets/patrycjawegrzynowicz/abc-factorized-attention-benchmark"
)


def load_csv(path: str | Path) -> pd.DataFrame:
    return pd.read_csv(path, keep_default_na=False)


def normalize_result_runs(runs: Any) -> pd.DataFrame:
    results_df = runs.as_dataframe().copy()
    if "result" not in results_df.columns:
        raise ValueError("Expected a 'result' column in runs.as_dataframe() output.")
    flat = pd.json_normalize(results_df["result"]).copy()
    for col in ("model", "model_name", "llm"):
        if col in results_df.columns and col not in flat.columns:
            flat[col] = results_df[col].values
    return flat


def parse_gold_lines(gold_lines: str) -> list[int]:
    value = json.loads(gold_lines)
    if not isinstance(value, list):
        raise ValueError("gold_lines must decode to a JSON list")
    return [int(x) for x in value]


def classify_prompt_error(exc: Exception) -> tuple[str, str]:
    error = str(exc)
    msg = error.lower()

    if "validation error" in msg or "pydantic" in msg or "list_type" in msg or "schema" in msg:
        return "parse_or_schema_error", error
    if "timeout" in msg:
        return "timeout_error", error
    if "connection" in msg or "connectivity" in msg or "api connection" in msg:
        return "connection_error", error
    if "rate limit" in msg or "429" in msg:
        return "rate_limit_error", error
    return "other_error", error


def print_overall_report(name: str, result_df: pd.DataFrame) -> None:
    passed = int(result_df["is_correct"].astype(int).sum())
    total = int(result_df["is_correct"].astype(int).count())
    acc = passed / total if total else 0.0
    print(f"{name}: {passed}/{total} = {acc:.2%}")


def summarize_accuracy(result_df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    return (
        result_df.groupby(group_cols)["is_correct"]
        .agg(["sum", "count", "mean"])
        .rename(columns={"sum": "passed", "count": "total", "mean": "accuracy"})
        .reset_index()
    )


def summarize_errors(result_df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    if "has_error" not in result_df.columns:
        return pd.DataFrame()
    return (
        result_df.groupby(group_cols)["has_error"]
        .agg(["sum", "count", "mean"])
        .rename(columns={"sum": "errors", "count": "total", "mean": "error_rate"})
        .reset_index()
    )


def summarize_error_types(result_df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    if "error_type" not in result_df.columns:
        return pd.DataFrame()
    errored = result_df[result_df["error_type"].fillna("") != ""]
    if errored.empty:
        return pd.DataFrame()
    return errored.groupby(group_cols + ["error_type"]).size().reset_index(name="count")


def print_group_reports(result_df: pd.DataFrame, *, task_name: str, groupings: list[list[str]]) -> None:
    print_overall_report(task_name, result_df)
    for cols in groupings:
        if any(c not in result_df.columns for c in cols):
            continue
        print(f"=== {task_name}: by {', '.join(cols)} ===")
        print(summarize_accuracy(result_df, cols).to_string(index=False))


def print_error_reports(result_df: pd.DataFrame, *, task_name: str, groupings: list[list[str]]) -> None:
    if "has_error" not in result_df.columns:
        return

    base_cols = [["model"]] if "model" in result_df.columns else []
    for cols in base_cols + groupings:
        if any(c not in result_df.columns for c in cols):
            continue

        error_summary = summarize_errors(result_df, cols)
        if not error_summary.empty:
            print(f"=== {task_name}: errors by {', '.join(cols)} ===")
            print(error_summary.to_string(index=False))

        error_type_summary = summarize_error_types(result_df, cols)
        if not error_type_summary.empty:
            print(f"=== {task_name}: error types by {', '.join(cols)} ===")
            print(error_type_summary.to_string(index=False))


def print_failures(result_df: pd.DataFrame, *, columns: list[str], max_rows: int = 20) -> None:
    failures = result_df.loc[~result_df["is_correct"].astype(bool), columns].copy()
    print(f"Failures shown: {min(len(failures), max_rows)}/{len(failures)}")
    if len(failures):
        print(failures.head(max_rows).to_string(index=False))


def available_columns(df: pd.DataFrame, cols: list[str]) -> list[str]:
    return [c for c in cols if c in df.columns]


def build_text_groupings(df: pd.DataFrame) -> list[list[str]]:
    groupings: list[list[str]] = [
        ["regime"],
        ["regime", "regime_level"],
        ["structure_type"] if "structure_type" in df.columns else ["target_feature_count"],
        ["num_records"],
    ]

    if "structure_depth" in df.columns:
        groupings.append(["structure_depth"])
        if "structure_type" in df.columns:
            groupings.append(["structure_type", "structure_depth"])
    if "binding_distance" in df.columns:
        groupings.append(["binding_distance"])
    if "serialization_style" in df.columns:
        groupings.append(["serialization_style"])
    if "position_mode" in df.columns:
        groupings.append(["position_mode"])
    if "target_count" in df.columns:
        groupings.append(["target_count"])
        groupings.append(["regime", "target_count"])
    if "confound_count" in df.columns:
        groupings.append(["confound_count"])
    if "confound_type" in df.columns:
        groupings.append(["confound_type"])
        if "structure_type" in df.columns:
            groupings.append(["structure_type", "confound_type"])
    if "line_length_noise" in df.columns:
        groupings.append(["line_length_noise"])

    # Deduplicate while preserving order.
    deduped: list[list[str]] = []
    seen: set[tuple[str, ...]] = set()
    for cols in groupings:
        key = tuple(cols)
        if key not in seen and all(col in df.columns for col in cols):
            seen.add(key)
            deduped.append(cols)
    return deduped


def run_dataset_single_model(
    *,
    item_task: Any,
    llm: Any,
    eval_df: pd.DataFrame,
    task_name: str,
    n_jobs: int = 1,
    groupings: list[list[str]] | None = None,
) -> tuple[pd.DataFrame, dict[str, int | float]]:
    runs = item_task.evaluate(llm=[llm], evaluation_data=eval_df, n_jobs=n_jobs)
    flat = normalize_result_runs(runs)
    if groupings:
        print_group_reports(flat, task_name=task_name, groupings=groupings)
        print_error_reports(flat, task_name=task_name, groupings=groupings)
    passed = int(flat["is_correct"].astype(int).sum())
    total = int(flat["is_correct"].astype(int).count())
    return flat, {"passed": passed, "total": total, "accuracy": (passed / total if total else 0.0)}